In [3]:
import pandas as pd
transactions = pd.read_csv("C:/Users/Diya/Desktop/fraud_ml/data/train_transaction.csv")
identity = pd.read_csv("C:/Users/Diya/Desktop/fraud_ml/data/train_identity.csv")
print(transactions.shape)
print(identity.shape)


(590540, 394)
(144233, 41)


In [4]:
df = transactions.merge(identity, on="TransactionID", how="left")
print(df.shape)

(590540, 434)


In [5]:
print(df['isFraud'].value_counts())


isFraud
0    569877
1     20663
Name: count, dtype: int64


In [6]:
print(df['isFraud'].value_counts(normalize=True))


isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64


In [8]:
null_percent = df.isnull().mean()
cols_to_drop = null_percent[null_percent > 0.9].index

df = df.drop(columns=cols_to_drop)

print("Remaining columns:", df.shape[1])


Remaining columns: 422


In [9]:
y = df['isFraud']
X = df.drop(['isFraud', 'TransactionID'], axis=1)


In [11]:
cat_cols = X.select_dtypes(include=['object', 'string', 'category']).columns


In [12]:
from sklearn.preprocessing import LabelEncoder
for col in cat_cols:
    X[col] = X[col].astype(str)
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])


In [13]:
X = X.fillna(-999)


In [14]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
)
model.fit(X_train, y_train)
preds = model.predict_proba(X_test)[:,1]
print("ROC-AUC:", roc_auc_score(y_test, preds))


ROC-AUC: 0.9467138568247363


In [16]:
import os
import joblib

# Create models directory if it doesn't exist
os.makedirs("models", exist_ok=True)

joblib.dump(model, "models/baseline_xgb.pkl")

print("Model saved successfully 🚀")


Model saved successfully 🚀
